# 7.2. Convolutions for Images
D2L의 Convolutions for Images장을 PyTorch 기준으로 정리함.

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [1]:
%matplotlib inline

import matplotlib.pyplot as plt 
import torch 
from torch import nn 
from torch.utils.data import DataLoader 
from torchvision import datasets, transforms

torch.manual_seed(42) 

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. 이미지에서 합성곱

CNN에선 이미지를 한 번에 완전연결층에 넣는 대신, 작은 크기의 kernel을 이미지의 여러 위치에 적용한다.

예를 들어서 이미지가 이럴때

```text
0 1 2
3 4 5
6 7 8

커널은
0 1
2 3
```
이러면 2 x 2 커널을 이미지 왼쪽 위부터 조금씩 이동시키며 계산한다.

입력 이미지 X + 커널 K => 지역별 특징 계산 => 출력 Feature Map

이게 CNN의 가장 기본적인 연산이다.

## 2. 2차원 Cross-Correlation

딥러닝에서 합성곱(convolution)이라고 부르는 연산은 엄밀하게 말하면 대부분 cross-correlation이다.

계산 방법은 이렇다.

1. 커널 크기만큼 이미지 일부를 잘라낸다.
2. 이미지 조각과 커널을 같은 위치끼리 곱한다.
3. 모든 값을 더한다.
4. 결과 하나를 출력한다.
5. 커널을 옆으로 이동한다.

## 3. 합성곱 직접 계산해보기

```text
입력 X = 
0 1 2
3 4 5
6 7 8

커널 K =
0 1
2 3
```

첫번째 위치에선 입력의 왼쪽 위 2 x 2를 본다. 커널이랑 위치별로 곱하면

0x0 + 1x1 + 3x2 + 4x3 = 19

이런식으로 계산을 하면
```text
19 25
37 43
```

이 나온다.

작은 영역 x 커널 => 위치별 곱 => 모두 더함 => 숫자 하나

이걸 이미지 전체에 반복한다.

## 4. 출력 크기는 어떻게 결정되는가?

Padding이나 Stride를 아직 사용하지 않으면 출력 크기는 이렇다.

입력 크기
$$
n_h​×n_w​
$$

커널 크기
$$
k_h​×k_w​
$$

출력 크기
$$
(n_h​−k_h​+1)×(n_w​−k_w​+1)
$$

예를 들어서 

입력 = 3 x 3, 커널 = 2 x 2 이면

높이 = 3 - 2 + 1 = 2, 너비 = 3 - 2 + 1 = 2 이다.

출력은 2 x 2가 된다. 커널이 이미지 바깥으로 나갈 수 없기 때문에 출력 크기가 작아지는 것이다.

## 5. Cross-Correlation 직접 구현

In [ ]:
def corr2d(X, K):
    h, w = K.shape

    output_h = X.shape[0] - h + 1
    output_w = X.shape[1] - w + 1

    Y = torch.zeros((output_h, output_w))

    for i in range(output_h):
        for j in range(output_w):
            region = X[i:i+h, j:j+w]     # 커널과 곱하기
            Y[i, j] = (region * K).sum() # 더하기

    return Y

In [3]:
X = torch.tensor([
    [0., 1., 2.],
    [3., 4., 5.],
    [6., 7., 8.]
])

K = torch.tensor([
    [0., 1.],
    [2., 3.]
])

corr2d(X, K)

tensor([[19., 25.],
        [37., 43.]])

## 6. Convolution Layer란?

합성곱층도 결국 전에 배운 `nn.Linear`랑 마찬가지로 학습 가능한 파라미터를 가진 Layer다.

완전 연결층에선

$$
Y = XW + b
$$

였다면, 합성곱층에선 개념적으로

$$
Y = X * K + b
$$
라고 생각할 수 있다.

* X = 입력 이미지
* K = kernel
* b = bias
* Y = 출력

커널은 처음부터 사람이 정해주는게 아니라 일반적으로 랜덤하게 초기화되고, 학습 과정에서 gradient descent를 통해 수정된다.

In [ ]:
class MyConv2D(nn.Module):

    def __init__(self, kernel_size):
        super().__init__()

        self.weight = nn.Parameter(
            torch.rand(kernel_size)
        )

        self.bias = nn.Parameter(
            torch.zeros(1)
        )

    def forward(self, X):
        return corr2d(X, self.weight) + self.bias # self.weight가 학습되는 kernel이다.

## 7. Kernel과 Filter는 무엇일까?

Kernel = 이미지의 작은 영역에 적용되는 학습 가능한 가중치 행렬 이라고 생각하면 된다.

예를 들어
```text
1 0 -1
1 0 -1
1 0 -1
```
이 kernel을 이미지 전체에 적용하면 Feature Map이 만들어진다. 설명에서는 kernel과 filter를 비슷한 의미로 말하는 경우가 많다.

그런데 RGB처럼 여러 input channel이 등장하면 조금 더 정확한 구분이 필요하다. 일단은 이렇게 생각하는게 편하다.

    Filter/Kernel = 어떤 특징을 찾기 위한 작은 가중치 집합

## 8. 합성곱으로 Edge 찾기

합성곱으로 무엇을 찾을 수 있는지 확인해보자.

6 x 8 이미지가 있다고 할때

In [5]:
X = torch.ones((6, 8))

X[:, 2:6] = 0

X

tensor([[1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.]])

가운데 0 을 검은색이라고 생각하면 된다. 이제 다음 kernel을 사용한다.

In [6]:
K = torch.tensor([
    [1., -1.]
])

    왼쪽 값 - 오른쪽 값을 계산한다.

## 9. 왜 [1, -1]이 Edge를 찾을까?

예를 들어 주변 픽셀 값이 같으면 

    1 1
에다가 

    1 -1

을 적용하면

1x1 + 1x(-1) = 0

변화가 없다.

    1 0
이면 

1x1 + 0x(-1) = 1
이다. 반대로 
    0 1
이면 

0x1 + 1x(-1) = -1
이다.

정리하면

```text
같은 색     => 0
1 => 0 변화 => +1
0 => 1 변화 => -1
```

픽셀이 갑자기 변하는 곳인 Edge를 찾을 수 있다.

In [7]:
Y = corr2d(X, K)

Y

tensor([[ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.]])

## 10. Kernel을 사람이 만드나?

Edge를 찾고 싶으면 [1, -1]을 넣으면 된다. 간단한 edge detector정도는 가능하다. 하지만 실제 CNN에는 수십~수백 개의 kernel이 있고, 여러 convolution layer가 겹쳐 있다.

각 kernel이 어떤 값을 가져야 하는지, 무슨 특징을 찾아야 하는지 사람이 직접 설계하는 건 불가능하다.

그래서 neural network가 kernel 자체를 학습한다.

## 11. Kernel 학습하기

CNN은 일반적으로 데이터를 이런 형태로 받는다

    [batch, channel, height, width]

In [ ]:
conv2d = nn.LazyConv2d(
    out_channels=1,
    kernel_size=(1, 2),
    bias=False
)

In [9]:
X_train = X.reshape(1, 1, 6, 8)
Y_train = Y.reshape(1, 1, 6, 7)

In [10]:
lr = 0.03

for epoch in range(10):

    Y_hat = conv2d(X_train)

    loss = ((Y_hat - Y_train) ** 2).sum()

    conv2d.zero_grad()
    loss.backward()

    with torch.no_grad():
        conv2d.weight -= lr * conv2d.weight.grad

    if (epoch + 1) % 2 == 0:
        print(
            f"epoch {epoch + 1}, "
            f"loss {loss.item():.4f}"
        )

epoch 2, loss 17.3498
epoch 4, loss 5.8622
epoch 6, loss 2.1924
epoch 8, loss 0.8630
epoch 10, loss 0.3476


우리가 목표로 하는 kernel은

[1, -1]
이었다. 

In [11]:
conv2d.weight.data.reshape(1, 2)

tensor([[ 1.0487, -0.9277]])

모델이 데이터만 보고 어떤 kernel을 사용해야 하는지 스스로 찾아낸 것이다. 

```text
랜덤 kernel
    ↓
forward
    ↓
prediction
    ↓
loss
    ↓
backward
    ↓
kernel gradient
    ↓
kernel 수정
```

## 12. Cross-Correlation과 Convolution

수학적으로 엄밀한 convolution은 kernel을

`좌우 반전 + 상하 반전` 하고 계산한다. 반면 딥러닝 프레임워크에서는 보통 kernel을 뒤집지 않은 cross-correlation을 사용한다.

하지만 neural network에선 kernel 자체를 학습하기 때문에 실제 모델링 관점에서는 큰 문제가 되지 않는다. 그래서 딥러닝 분야에는 관습적으로 cross-correlation 연산도 convolution이라고 부른다고 한다.

## 13. Feature Map과 Receptive Field

합성곱을 적용해 나온 출력은 보통 Feature Map이라고 부른다.

```text
Input Image
     ↓
Kernel
     ↓
Convolution
     ↓
Feature Map
```

예를 들어 edge detector kenel을 사용했다면 feature map에는 Edge가 있는 위치가 강하게 나타난다.

Feature Map은 해당 kernel이 찾고 있는 특징이 이미지의 어디에 존재하는지를 표현한 결과라고 생각하면 된다.

Receptive Field(수용영역)는 출력의 한 값이 입력 이미지의 어느 영역을 보고 계산되었는지를 의미한다.

예를 들어서

2 x 2 kernel이면 출력 한 칸은 처음 입력의 2 x 2를 본다.

convolution layer가 여러 층 쌓이면 더 깊은 layer의 뉴런은 원본 이미지의 훨씬 넓은 영역의 영향을 받게 된다.

## 14. 전체 흐름

```text
이미지 X
   ↓
작은 Kernel K를 올림
   ↓
같은 위치끼리 곱함
   ↓
모두 더함
   ↓
숫자 하나 생성
   ↓
Kernel 이동
   ↓
반복
   ↓
Feature Map 생성
```

실제 CNN은 kernel을 학습한다.
```text
Random Kernel
      ↓
Forward
      ↓
Loss
      ↓
Backward
      ↓
Gradient
      ↓
Kernel 수정
```

```text
Linear => 모든 입력과 연결

Convolution => 작은 지역에 같은 kernel을 반복 적용
```

## 15. 오늘의 정리

- 합성곱은 작은 kernel을 이미지 위에서 움직이며 계산하는 연산이다.
- Kernel 영역과 이미지 영역을 위치별로 곱한 뒤 모두 더해서 출력 값 하나를 만든다.
- 이 작업을 반복하면 Feature Map이 만들어진다.
- 출력 크기는 기본적으로 (입력 - kernel + 1)로 계산된다.
- Kernel은 CNN의 학습 가능한 weight다.
- [1, -1] 같은 kernel은 인접 픽셀의 차이를 계산해서 edge를 검출할 수 있다.
- 실제 CNN에서는 사람이 kernel을 설계하지 않고 backpropagation으로 kernel을 학습한다.
- 딥러닝에서 convolution이라고 부르는 연산은 엄밀하게는 대부분 cross-correlation이다.
- Feature Map은 kernel이 찾은 특징이 어디에 존재하는지를 표현한 출력이다.
- Receptive Field는 출력 값 하나를 계산할 때 영향을 주는 입력 영역이다.